In [1]:
import carla
import math
import random
import time
import numpy as np
import cv2

client = carla.Client('localhost', 2000)
client.set_timeout(10.0)

world = client.get_world()
traffic_manager = client.get_trafficmanager()
traffic_manager.set_synchronous_mode(True)
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

bp_lib = world.get_blueprint_library()
spawn_points = world.get_map().get_spawn_points()

print("Setup cell ran succesfully")

Setup cell ran succesfully


In [2]:
vehicle_bp = bp_lib.find('vehicle.tesla.model3')
vehicle = world.try_spawn_actor(vehicle_bp, random.choice(spawn_points))
#vehicle.show_debug_telemetry(True)
world.tick()  # sync mode: advance one step so the vehicle actually appears

spectator = world.get_spectator()

IMAGE_W = 1600
IMAGE_H = 900

def make_camera_bp(fov):
    bp = bp_lib.find('sensor.camera.rgb')
    bp.set_attribute('image_size_x', str(IMAGE_W))
    bp.set_attribute('image_size_y', str(IMAGE_H))
    bp.set_attribute('fov', str(fov))
    return bp

# 6 camera suite setup: (name, transform, fov)
# note: back_right uses a wider 110 fov, the rest use 70
camera_specs = [
    ('front',       carla.Transform(carla.Location(z=1.5)),                            70),   # front middle
    ('front_left',  carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=-55)),   70),   # front left
    ('front_right', carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+55)),   70),   # front right
    ('back',        carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+180)),  70),   # back middle
    ('back_left',   carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=-120)),  70),   # back left
    ('back_right',  carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+120)),  110),  # back right
]

In [3]:
# spawn all 6 cameras at the same time, attached to the vehicle
cameras = {}
for name, trans, fov in camera_specs:
    cameras[name] = world.spawn_actor(make_camera_bp(fov), trans, attach_to=vehicle)

print('spawned', len(cameras), 'cameras:', list(cameras))

# park the spectator on the front camera so the CARLA window has a useful view
spectator.set_transform(cameras['front'].get_transform())
world.tick()  # sync mode: advance one step so the cameras register and the spectator move renders

spawned 6 cameras: ['front', 'front_left', 'front_right', 'back', 'back_left', 'back_right']


568

In [4]:
import csv
import os

# ---- synchronized multi-camera dataset layout -------------------------------
# dataset/
#   FRONT/<frame>.png  FRONT_LEFT/<frame>.png  ...  BACK_RIGHT/<frame>.png
#   output.csv         one row per frame; frame_id is the synchronization key that
#                      links the row to <frame_id>.png in every camera folder
#                      (manual step 3: frame 150 in the csv -> 00150.png).
OUTPUT_DIR = 'dataset'
folder_for = {name: name.upper() for name in cameras}   # 'front' -> 'FRONT', etc.
for folder in folder_for.values():
    os.makedirs(os.path.join(OUTPUT_DIR, folder), exist_ok=True)

# standard column format shared by all groups (matches example_output.csv).
# per the manual: 'speed' is in km/h and 'timestamp' is simulation time in seconds.
fieldnames = ['frame_id', 'timestamp', 'speed', 'input_throttle', 'input_steer',
              'input_brake', 'x', 'y', 'z', 'pitch', 'yaw', 'roll']

csv_file = open(os.path.join(OUTPUT_DIR, 'output.csv'), 'w', newline='')
csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
csv_writer.writeheader()

# latest frame per camera, kept only for the opencv preview windows
camera_data = {name: np.zeros((IMAGE_H, IMAGE_W, 4), dtype=np.uint8) for name in cameras}

In [ ]:
import json

# ---- camera calibration (written once; constant for the whole run) ----------
# intrinsics depend only on resolution + fov, and the camera->vehicle extrinsic is
# the static mounting transform. camera->world for any frame is then:
#   T_cam2world = T_vehicle2world(frame) @ T_cam2vehicle
# where T_vehicle2world comes from the ego pose (x,y,z,pitch,yaw,roll) in output.csv.
# all values use CARLA's convention: x forward, y right, z up, metres/degrees.

def intrinsic_matrix(width, height, fov_deg):
    f = width / (2.0 * math.tan(math.radians(fov_deg) / 2.0))  # square pixels, horizontal fov
    cx, cy = width / 2.0, height / 2.0
    return [[f,   0.0, cx],
            [0.0, f,   cy],
            [0.0, 0.0, 1.0]]

calibration = {}
for name, trans, fov in camera_specs:
    loc, rot = trans.location, trans.rotation
    calibration[folder_for[name]] = {
        'image_size': [IMAGE_W, IMAGE_H],
        'fov': fov,
        'intrinsic': intrinsic_matrix(IMAGE_W, IMAGE_H, fov),
        # static 4x4 camera->vehicle extrinsic (row-major homogeneous matrix)
        'extrinsic_cam_to_vehicle': trans.get_matrix(),
        # same mounting in human-readable form
        'mounting': {
            'location': [loc.x, loc.y, loc.z],
            'rotation': {'pitch': rot.pitch, 'yaw': rot.yaw, 'roll': rot.roll},
        },
    }

with open(os.path.join(OUTPUT_DIR, 'calibration.json'), 'w') as cal_file:
    json.dump(calibration, cal_file, indent=2)
print('wrote', os.path.join(OUTPUT_DIR, 'calibration.json'))

In [5]:
import queue

# one queue per camera. in synchronous mode every world.tick() pushes exactly one
# image per camera, and all six images share the same frame id. matching on that
# frame id in the capture loop is what bundles a perfectly synchronized snapshot.
image_queues = {name: queue.Queue() for name in cameras}
for name, cam in cameras.items():
    cam.listen(image_queues[name].put)

In [6]:
# bind to the same Traffic Manager that was put into synchronous mode in the setup cell
vehicle.set_autopilot(True, traffic_manager.get_port())

# one window per camera, tiled in a 3x2 grid that mirrors the physical layout
# so the windows don't open stacked on top of each other.
#   row 0:  Front Left | Front | Front Right
#   row 1:  Back Left  | Back  | Back Right
titles = {
    'front_left': 'Front Left',  'front': 'Front',  'front_right': 'Front Right',
    'back_left':  'Back Left',   'back':  'Back',    'back_right':  'Back Right',
}
grid = {
    'front_left': (0, 0),  'front': (1, 0),  'front_right': (2, 0),
    'back_left':  (0, 1),  'back':  (1, 1),  'back_right':  (2, 1),
}

THUMB_W, THUMB_H = 480, 270   # display size per window (16:9, scaled down from 1600x900)
GAP_X, GAP_Y = 12, 48         # spacing between windows; GAP_Y leaves room for title bars

for name, (col, row) in grid.items():
    win = titles[name]
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, THUMB_W, THUMB_H)
    cv2.moveWindow(win, col * (THUMB_W + GAP_X), row * (THUMB_H + GAP_Y))

recording = False
recorded = 0
print("press 'r' to toggle recording, 'q' to quit")

while True:
    # in synchronous mode we own the clock: advance the sim one fixed step.
    # world.tick() returns the id of the frame it just produced.
    world_frame = world.tick()

    # bundle one synchronized snapshot: from every camera pull the image whose
    # frame matches THIS tick. a queue may briefly hold an older frame, so discard
    # until we reach world_frame; the timeout surfaces a stalled sensor rather than
    # hanging forever.
    snapshot = {}
    for name, q in image_queues.items():
        while True:
            image = q.get(timeout=2.0)
            if image.frame == world_frame:
                snapshot[name] = image
                break

    # live preview of all six cameras
    for name, image in snapshot.items():
        camera_data[name] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))
    for name in grid:
        cv2.imshow(titles[name], camera_data[name])

    # record one synchronized sample: six images + one telemetry row, all keyed by
    # frame_id. the image filename matches the csv frame_id exactly (manual step 3),
    # so frame 180692 in output.csv <-> 180692.png in every camera folder.
    if recording:
        for name, image in snapshot.items():
            path = os.path.join(OUTPUT_DIR, folder_for[name], f'{world_frame:06d}.png')
            image.save_to_disk(path)

        snap = world.get_snapshot()
        control = vehicle.get_control()
        transform = vehicle.get_transform()
        velocity = vehicle.get_velocity()
        loc, rot = transform.location, transform.rotation
        speed_kmh = 3.6 * math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)  # m/s -> km/h

        csv_writer.writerow({
            'frame_id': world_frame,
            'timestamp': snap.timestamp.elapsed_seconds,  # simulation time in seconds
            'speed': speed_kmh,
            'input_throttle': control.throttle,
            'input_steer': control.steer,
            'input_brake': control.brake,
            'x': loc.x,
            'y': loc.y,
            'z': loc.z,
            'pitch': rot.pitch,
            'yaw': rot.yaw,
            'roll': rot.roll,
        })
        csv_file.flush()  # crash-safe: each row hits disk
        recorded += 1

    key = cv2.waitKey(1) & 0xFF
    if key == ord('r'):
        recording = not recording
        print(('recording, ' if recording else 'paused, ') + f'{recorded} frames saved so far')
    elif key == ord('q'):
        break

cv2.destroyAllWindows()

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tygo/code/vla_research/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
N

press 'r' to toggle recording, 'q' to quit
recording, 0 frames saved so far
paused, 127 frames saved so far


In [ ]:
# stop logging and close the file (run after stopping the loop above)
for cam in cameras.values():
    cam.stop()
csv_file.close()

# restore asynchronous mode so the server keeps running on its own once we stop
# ticking (otherwise the sim freezes waiting for the next world.tick()).
traffic_manager.set_synchronous_mode(False)
settings = world.get_settings()
settings.synchronous_mode = False
settings.fixed_delta_seconds = None
world.apply_settings(settings)

print(f'telemetry + images written under {OUTPUT_DIR}/')